# Перенос художественного стиля для мобильного устройства



In [2]:
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/style_transfer_project')
WIKIART_ARCHIVE = PROJECT_DIR / 'wikiart.tar'
DATA_DIR = PROJECT_DIR / 'data'
WIKIART_DIR = DATA_DIR / 'wikiart'
STYLE_DIR = DATA_DIR / 'style_images'
CONTENT_DIR = DATA_DIR / 'content_images'
MODELS_DIR = PROJECT_DIR / 'models'
RESULTS_DIR = PROJECT_DIR / 'results'
REPORTS_DIR = PROJECT_DIR / 'reports'
for directory in [PROJECT_DIR, DATA_DIR, WIKIART_DIR, STYLE_DIR, CONTENT_DIR, MODELS_DIR, RESULTS_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print('Архив:', WIKIART_ARCHIVE)
print('Архив существует:', WIKIART_ARCHIVE.exists())


Архив: \content\drive\MyDrive\style_transfer_project\wikiart.tar
Архив существует: False


In [3]:
!pip install -q tensorflow==2.15.1 tensorflow-model-optimization pillow matplotlib pandas opencv-python tqdm


  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: Could not install packages due to an OSError: [WinError 5] Отказано в доступе: 'c:\\Users\\aleks\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\tensorflow\\compiler\\mlir\\quantization\\tensorflow\\python\\pywrap_quantize_model.pyd'
Consider using the `--user` option or check the permissions.



In [4]:
import random, shutil, tarfile, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image, ImageFile
from tqdm.auto import tqdm
ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('TensorFlow:', tf.__version__)


ImportError: DLL load failed while importing _pywrap_tf2: Не найден указанный модуль.

## Распаковка и проверка WikiArt


In [ ]:
def extract_tar_archive(archive_path, output_dir):
    archive_path, output_dir = Path(archive_path), Path(output_dir)
    if not archive_path.exists():
        raise FileNotFoundError(f'Архив не найден: {archive_path}')
    output_dir.mkdir(parents=True, exist_ok=True)
    if any(output_dir.iterdir()):
        print('Архив уже был распакован.')
        return
    with tarfile.open(archive_path, 'r:*') as tar:
        tar.extractall(output_dir)
    print('Распаковка завершена.')
extract_tar_archive(WIKIART_ARCHIVE, WIKIART_DIR)


In [ ]:
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
def find_image_files(directory):
    return sorted([p for p in Path(directory).rglob('*') if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS])
def is_valid_image(path, min_width=128, min_height=128):
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            w, h = image.size
        return w >= min_width and h >= min_height
    except Exception:
        return False
wikiart_paths = find_image_files(WIKIART_DIR)
valid_wikiart_paths = [p for p in tqdm(wikiart_paths, desc='Проверка WikiArt') if is_valid_image(p)]
print('Найдено:', len(wikiart_paths))
print('Валидно:', len(valid_wikiart_paths))
if not valid_wikiart_paths:
    raise ValueError('В архиве не найдено валидных изображений.')
pd.DataFrame({'path':[str(p) for p in valid_wikiart_paths]}).to_csv(REPORTS_DIR / 'wikiart_valid_images.csv', index=False)


In [ ]:
def show_images(paths, cols=4, figsize=(16, 12)):
    paths = list(paths)
    rows = int(np.ceil(len(paths) / cols))
    plt.figure(figsize=figsize)
    for i, path in enumerate(paths, 1):
        plt.subplot(rows, cols, i)
        plt.imshow(Image.open(path).convert('RGB'))
        plt.title(Path(path).name[:30], fontsize=9)
        plt.axis('off')
    plt.tight_layout()
    plt.show()
show_images(random.sample(valid_wikiart_paths, min(12, len(valid_wikiart_paths))))


## Подготовка стилевых и контентных изображений


In [ ]:
N_STYLE_IMAGES = 30
selected_style_paths = random.sample(valid_wikiart_paths, min(N_STYLE_IMAGES, len(valid_wikiart_paths)))
def copy_style_images(source_paths, destination_dir):
    result = []
    for i, source in enumerate(source_paths):
        target = Path(destination_dir) / f'style_{i:03d}{source.suffix.lower()}'
        shutil.copy2(source, target)
        result.append(target)
    return result
style_paths = copy_style_images(selected_style_paths, STYLE_DIR)
print('Стилевых изображений:', len(style_paths))
show_images(style_paths[:12])


In [ ]:
content_paths = find_image_files(CONTENT_DIR)
print('Контентных изображений:', len(content_paths))
if not content_paths:
    raise ValueError('Добавьте фотографии в data/content_images на Google Drive, затем перезапустите эту ячейку.')
show_images(content_paths[:min(12, len(content_paths))])


## Функции обработки изображений


In [ ]:
def load_image_tensor(path, image_size):
    image = tf.io.read_file(str(path))
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, image_size)
    return tf.expand_dims(image, 0)
def show_tensor_image(image, title=None):
    image = image.numpy() if isinstance(image, tf.Tensor) else image
    if image.ndim == 4:
        image = image[0]
    plt.figure(figsize=(6,6))
    plt.imshow(np.clip(image, 0, 1))
    plt.axis('off')
    if title: plt.title(title)
    plt.show()
def save_tensor_image(image, path):
    image = image.numpy() if isinstance(image, tf.Tensor) else image
    if image.ndim == 4: image = image[0]
    Image.fromarray((np.clip(image, 0, 1)*255).astype(np.uint8)).save(path)
    print('Сохранено:', path)


## Загрузка SavedModel Magenta


In [ ]:
SAVED_MODELS_DIR = Path('/content/saved_models')
SAVED_MODELS_DIR.mkdir(exist_ok=True)
!wget -q -O /content/style_predict.tar.gz https://storage.googleapis.com/download.tensorflow.org/models/tflite/style_predict.tar.gz
!wget -q -O /content/style_transform.tar.gz https://storage.googleapis.com/download.tensorflow.org/models/tflite/style_transform.tar.gz
!tar -xzf /content/style_predict.tar.gz -C /content/saved_models
!tar -xzf /content/style_transform.tar.gz -C /content/saved_models
saved_model_dirs = [p.parent for p in SAVED_MODELS_DIR.rglob('saved_model.pb')]
print(*saved_model_dirs, sep='\n')
STYLE_PREDICT_SAVED_MODEL = SAVED_MODELS_DIR / 'style_predict'
STYLE_TRANSFORM_SAVED_MODEL = SAVED_MODELS_DIR / 'style_transform'
print('style_predict:', STYLE_PREDICT_SAVED_MODEL.exists())
print('style_transform:', STYLE_TRANSFORM_SAVED_MODEL.exists())


## Конвертация в TensorFlow Lite


In [ ]:
STYLE_PREDICT_TFLITE = MODELS_DIR / 'style_predict.tflite'
STYLE_TRANSFORM_FLOAT32 = MODELS_DIR / 'style_transform_float32.tflite'
STYLE_TRANSFORM_DYNAMIC = MODELS_DIR / 'style_transform_dynamic.tflite'
STYLE_TRANSFORM_FLOAT16 = MODELS_DIR / 'style_transform_float16.tflite'
STYLE_TRANSFORM_INT8 = MODELS_DIR / 'style_transform_int8.tflite'
def write_model(model, path):
    with open(path, 'wb') as f: f.write(model)
    print('Сохранена модель:', path)
def convert_model(saved_model_path, output_path, mode='float32'):
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_path))
    if mode == 'dynamic':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    elif mode == 'float16':
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    write_model(converter.convert(), output_path)
convert_model(STYLE_PREDICT_SAVED_MODEL, STYLE_PREDICT_TFLITE)
convert_model(STYLE_TRANSFORM_SAVED_MODEL, STYLE_TRANSFORM_FLOAT32)
convert_model(STYLE_TRANSFORM_SAVED_MODEL, STYLE_TRANSFORM_DYNAMIC, 'dynamic')
convert_model(STYLE_TRANSFORM_SAVED_MODEL, STYLE_TRANSFORM_FLOAT16, 'float16')


In [ ]:
def inspect_tflite_model(path):
    interpreter = tf.lite.Interpreter(model_path=str(path))
    interpreter.allocate_tensors()
    print('Модель:', Path(path).name)
    print('Входы:')
    for x in interpreter.get_input_details(): print(x['index'], x['name'], x['shape'], x['dtype'])
    print('Выходы:')
    for x in interpreter.get_output_details(): print(x['index'], x['name'], x['shape'], x['dtype'])
inspect_tflite_model(STYLE_PREDICT_TFLITE)
inspect_tflite_model(STYLE_TRANSFORM_FLOAT32)


## Инференс модели


In [ ]:
def get_style_bottleneck(model_path, style_image):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    inp, out = interpreter.get_input_details()[0], interpreter.get_output_details()[0]
    interpreter.set_tensor(inp['index'], style_image.numpy().astype(inp['dtype']))
    interpreter.invoke()
    return interpreter.get_tensor(out['index'])
def identify_transform_inputs(details):
    content = bottleneck = None
    for x in details:
        name = x['name'].lower()
        if 'content' in name or 'image' in name: content = x
        if 'style' in name or 'bottleneck' in name: bottleneck = x
    return (content, bottleneck) if content is not None and bottleneck is not None else (details[0], details[1])
def stylize_with_tflite(model_path, content_image, bottleneck):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    content, _ = identify_transform_inputs(interpreter.get_input_details())
    interpreter.resize_tensor_input(content['index'], content_image.shape, strict=False)
    interpreter.allocate_tensors()
    content, style = identify_transform_inputs(interpreter.get_input_details())
    output = interpreter.get_output_details()[0]
    interpreter.set_tensor(content['index'], content_image.numpy().astype(content['dtype']))
    interpreter.set_tensor(style['index'], bottleneck.astype(style['dtype']))
    interpreter.invoke()
    return interpreter.get_tensor(output['index'])
content_example = load_image_tensor(content_paths[0], (384,384))
style_example = load_image_tensor(style_paths[0], (256,256))
bottleneck_example = get_style_bottleneck(STYLE_PREDICT_TFLITE, style_example)
result_float32 = stylize_with_tflite(STYLE_TRANSFORM_FLOAT32, content_example, bottleneck_example)
show_tensor_image(content_example, 'Контент')
show_tensor_image(style_example, 'Стиль')
show_tensor_image(result_float32, 'Float32')
save_tensor_image(result_float32, RESULTS_DIR / 'example_float32.jpg')


## Int8-квантизация с representative dataset


In [ ]:
rep_styles = random.sample(style_paths, min(20, len(style_paths)))
rep_contents = random.sample(content_paths, min(20, len(content_paths)))
rep_bottlenecks = [get_style_bottleneck(STYLE_PREDICT_TFLITE, load_image_tensor(p, (256,256))).astype(np.float32) for p in tqdm(rep_styles)]
def representative_dataset():
    for path in rep_contents:
        content = load_image_tensor(path, (384,384)).numpy().astype(np.float32)
        yield [content, random.choice(rep_bottlenecks)]
try:
    converter = tf.lite.TFLiteConverter.from_saved_model(str(STYLE_TRANSFORM_SAVED_MODEL))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    write_model(converter.convert(), STYLE_TRANSFORM_INT8)
    int8_conversion_success = True
except Exception as error:
    int8_conversion_success = False
    print('Int8-конвертация не завершилась:', error)


## Сравнение размеров и скорости


In [ ]:
transform_models = {'Float32': STYLE_TRANSFORM_FLOAT32, 'Dynamic Range': STYLE_TRANSFORM_DYNAMIC, 'Float16': STYLE_TRANSFORM_FLOAT16}
if int8_conversion_success: transform_models['Int8'] = STYLE_TRANSFORM_INT8
def size_mb(path): return Path(path).stat().st_size/(1024*1024)
model_size_table = pd.DataFrame([{'Модель': n, 'Размер, MB': round(size_mb(p),3)} for n,p in transform_models.items()])
model_size_table.to_csv(REPORTS_DIR/'model_sizes.csv', index=False)
model_size_table


In [ ]:
def measure(model_path, runs=10):
    times=[]
    for _ in range(2): stylize_with_tflite(model_path, content_example, bottleneck_example)
    for _ in range(runs):
        start=time.perf_counter()
        stylize_with_tflite(model_path, content_example, bottleneck_example)
        times.append((time.perf_counter()-start)*1000)
    return {'Среднее, мс':round(float(np.mean(times)),2),'Медиана, мс':round(float(np.median(times)),2)}
rows=[]
for name, path in transform_models.items():
    row=measure(path)
    row['Модель']=name
    row['Размер, MB']=round(size_mb(path),3)
    rows.append(row)
timing_table=pd.DataFrame(rows)[['Модель','Размер, MB','Среднее, мс','Медиана, мс']]
timing_table.to_csv(REPORTS_DIR/'inference_time_comparison.csv',index=False)
timing_table


In [ ]:
results={name:stylize_with_tflite(path,content_example,bottleneck_example) for name,path in transform_models.items()}
plt.figure(figsize=(5*(2+len(results)),5))
plt.subplot(1,2+len(results),1); plt.imshow(content_example[0]); plt.title('Контент'); plt.axis('off')
plt.subplot(1,2+len(results),2); plt.imshow(style_example[0]); plt.title('Стиль'); plt.axis('off')
for i,(name,result) in enumerate(results.items(),3):
    plt.subplot(1,2+len(results),i); plt.imshow(np.clip(result[0],0,1)); plt.title(name); plt.axis('off')
    save_tensor_image(result, RESULTS_DIR/f'comparison_{name.lower().replace(" ","_")}.jpg')
plt.tight_layout()
plt.savefig(RESULTS_DIR/'quantization_comparison.png',dpi=150,bbox_inches='tight')
plt.show()


## Итоговый отчёт и архив артефактов


In [ ]:
summary = '# Итоговый отчёт\n\n'
summary += f'- Найдено изображений WikiArt: {len(wikiart_paths)}\n'
summary += f'- Валидных изображений WikiArt: {len(valid_wikiart_paths)}\n'
summary += f'- Стилевых изображений: {len(style_paths)}\n'
summary += f'- Контентных изображений: {len(content_paths)}\n\n'
summary += '## Размеры моделей\n\n' + model_size_table.to_markdown(index=False)
summary += '\n\n## Скорость инференса\n\n' + timing_table.to_markdown(index=False)
with open(REPORTS_DIR/'project_summary.md','w',encoding='utf-8') as f: f.write(summary)
print('Отчёт сохранён:', REPORTS_DIR/'project_summary.md')
EXPORT_DIR = Path('/content/style_transfer_export')
if EXPORT_DIR.exists(): shutil.rmtree(EXPORT_DIR)
EXPORT_DIR.mkdir()
for folder in [MODELS_DIR, RESULTS_DIR, REPORTS_DIR]:
    shutil.copytree(folder, EXPORT_DIR/folder.name, dirs_exist_ok=True)
ARCHIVE_PATH=PROJECT_DIR/'style_transfer_project_artifacts.zip'
shutil.make_archive(str(ARCHIVE_PATH).replace('.zip',''),'zip',EXPORT_DIR)
print('Архив создан:', ARCHIVE_PATH)
